In [33]:
import pandas as pd, glob

path = "/Users/nitishbhattad/Desktop/kafka-fraud-detection/data/all_transactions/*.parquet"
df = pd.concat([pd.read_parquet(f) for f in glob.glob(path)], ignore_index=True)
len(df)
df.head()


ArrowInvalid: Could not open Parquet input source '<Buffer>': Parquet file size is 0 bytes

In [ ]:
import os, glob

path = "/Users/nitishbhattad/Desktop/kafka-fraud-detection/data/all_transactions/*.parquet"
files = glob.glob(path)
len(files), files[:5]


In [ ]:
sizes = [(os.path.basename(f), os.path.getsize(f)) for f in files]
sizes[:10]


In [ ]:
import pandas as pd, glob, os

path = "/Users/nitishbhattad/Desktop/kafka-fraud-detection/data/all_transactions/*.parquet"
files = glob.glob(path)

df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

len(df), df.head()


In [ ]:
# start fresh labels
df['is_fraud'] = 0

# rule 1: high amount
df.loc[df['amount'] > 3000, 'is_fraud'] = 1

# rule 2: risky device
df.loc[df['device_type'] == 'Desktop', 'is_fraud'] = 1

df['is_fraud'].value_counts()


In [ ]:
import numpy as np

df['is_fraud'] = 0
df.loc[df['amount'] > 3000, 'is_fraud'] = 1
df.loc[df['device_type'] == 'Desktop', 'is_fraud'] = 1


In [40]:
X = df[['amount', 'device_type', 'merchant']]
y = df['is_fraud']


In [42]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import joblib

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['device_type', 'merchant'])
    ],
    remainder='passthrough'
)

model = Pipeline([
    ('preprocess', preprocess),
    ('clf', LogisticRegression(max_iter=1000))
])


In [44]:
model.fit(X_train, y_train)


Pipeline(steps=[('preprocess',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['device_type',
                                                   'merchant'])])),
                ('clf', LogisticRegression(max_iter=1000))])

In [46]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.94      1.00      0.97        16
           1       1.00      0.99      1.00       142

    accuracy                           0.99       158
   macro avg       0.97      1.00      0.98       158
weighted avg       0.99      0.99      0.99       158



In [48]:
features = [
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
]

target = "isFraud"


In [50]:
df_model = df[features + [target]].copy()
df_model.head()


KeyError: "['type', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud'] not in index"

In [52]:
from sklearn.model_selection import train_test_split

X = data[features].copy()
y = data[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape, y_train.mean(), y_test.mean()


NameError: name 'data' is not defined